In [1]:
import json
import pandas as pd
from tqdm import tqdm
import datasets
import os

/projects/iris/rferreir/.envs/spe/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Path to v2.1 of the train split
path_train = f'/projects/iris/rferreir/datasets/MsMarco/original_data/train_v2.1.json'

# Path to v2.1 of the dev split
path_dev = f'/projects/iris/rferreir/datasets/MsMarco/original_data/dev_v2.1.json'

# Path to the result.json file containing the analytics of the questions
result_path = '/projects/iris/rferreir/datasets/MsMarco/location_questions/result.json'

# Path to the directory containing the HF datasets 
output_path = "/projects/iris/rferreir/hub_datasets"

## Filtering the data

In [3]:
with open(path_train, 'r') as f:
    train = json.load(f)

with open(path_dev, 'r') as f:
    dev = json.load(f)
    
with open(result_path, 'r') as f:
    data = json.load(f)

In [4]:
# We will only keep questions with interrogative key words to avoir ambiguous questions
q_words = ['what', 'where', 'which', 'why', 'how', 'when']
count = {}
for w in q_words:
    count[w] = 0

for e in tqdm(data):
    for qw in count:
        if qw in e['query'].lower().split(" ") and len(e['queryAnalyze']['placeName']) > 0 and e['answers'][0] != 'No Answer Present.':
            count[qw] += 1

print(count)

100%|██████████| 56721/56721 [00:00<00:00, 315398.54it/s]

{'what': 14429, 'where': 15404, 'which': 797, 'why': 11, 'how': 12, 'when': 25}


In [5]:
i = 0
for e in tqdm(data):
    if 'how' in e['query'].lower().split(" ") and len(e['queryAnalyze']['placeName']) > 0 and e['answers'][0] != 'No Answer Present.':
        print(e['query'])
        print(e['answers'][0])
        print()
        i += 1
    if i == 5:
        break

 44%|████▍     | 24906/56721 [00:00<00:00, 1168702.85it/s]

idaho has how many counties
44 counties

norfolk airport is how close to hampton university
15 miles

the agreement over how states would be represented in congress was known as the
Constitutional convention

how is the location of the suez canal strategic
It is located in Egypt, and connects connects Port Said on the Mediterranean Sea with the port of Suez on the Red Sea.

where and how does olives grow
Olives can be grown relatively easily from a seed, or pit, however it must be noted that most olive trees grown from a seed,olives are native to warm temperate and tropical regions in Europe, Australasia, Africa, and southern Asia.



In [6]:
i = 0
for e in tqdm(data):
    if 'when' in e['query'].lower().split(" ") and len(e['queryAnalyze']['placeName']) > 0 and e['answers'][0] != 'No Answer Present.':
        print(e['query'])
        print(e['answers'][0])
        print()
        i += 1
    if i == 5:
        break

 32%|███▏      | 18398/56721 [00:00<00:00, 1273106.51it/s]

what was created when the towns of winston and salem combined
Winston-Salem was created when the two towns Winston and Salem combined.

where and when was james k polk death
James K. Polk died in Nashville, Tennessee.

what happened to the six counties of northern ireland when the republic of ireland was formed
After years of civil war, Ireland became a republic in 1921. At this time, Britain negotiated with Ireland to keep the six counties in the north-east of Ireland. These six counties now make up what is known as Northern Ireland or Ulster. 

where does the appalachian trail end when going north
Reaches an early high point atop windswept mt washington in the heart of new hampshire s presidential range.

where and when was john boyne born
John Boyne was born on April 30, 1971 in Dublin, Ireland.



In [7]:
i = 0
for e in tqdm(data):
    if 'which' in e['query'].lower().split(" ") and len(e['queryAnalyze']['placeName']) > 0 and e['answers'][0] != 'No Answer Present.':
        print(e['query'])
        print(e['answers'][0])
        print()
        i += 1
    if i == 5:
        break

  3%|▎         | 1651/56721 [00:00<00:00, 1290254.50it/s]

which allied nation controlled the suez canal in north africa?great britainegyptjapanthe United States
Great Britain is the allied nation controlled the suez canal in North Africa.

which borough is highgate in
Highgate is a ward in the London Borough of Camden, in the United Kingdom.

which brussels train station for luxembourg city
Brussels Central Station  is a metro and railway station in Brussels for Luxembourg city

which cities in florida is worst to live in
Kissimmee is the worst city to live in Florida.

which city suffered the most lost territory as a result of the unification of italy
The country that suffered the most lost territory as a result of the unification of Italy was Austria.



In [8]:
# We map the each query_id to its split
map_q_split = {}
for e in train['query_id']:
    idx, q, a, p = train['query_id'][e], train['query'][e], train['answers'][e], train['passages'][e]
    map_q_split[q.lower()] = {'id': idx, 'split': 'train', 'query': q, 'answer': a[0], 'passages': p}

for e in dev['query_id']:
    idx, q, a, p = dev['query_id'][e], dev['query'][e], dev['answers'][e], dev['passages'][e]
    map_q_split[q.lower()] = {'id': idx, 'split': 'test', 'query': q, 'answer': a[0], 'passages': p}
    
keep = {'question_id': [], 'question': [], 'answer': [], 'split': [], 'passages': [], "to_get_rid_of": []}

# We don't keep the 'How', 'when' and 'why' as their are not really viewed as geographic questions
q_words = set(['what', 'where', 'which'])
for e in tqdm(data):
    a = map_q_split[e['query'].lower()]['answer']
    # We keep questions that have at least one geographic place mentioned and also an answer
    if len(e['queryAnalyze']['placeName']) > 0 and a != 'No Answer Present.':
        q = set(e['query'].lower().split())
        if len(q_words.intersection(q)) > 0:
            rid_off = False
            # Sometimes the answer can be an empty string, we don't keep the questions then
            if a == "":
                rid_off = True
            query = e['query'].lower()
            idx = map_q_split[query]['id']
            split = map_q_split[query]['split']
            q = map_q_split[query]['query'].strip()
            p = map_q_split[query]['passages']

            q = q[0].upper() + q[1:] + " ?"
            keep['question_id'].append(idx)
            keep['question'].append(q)
            keep['answer'].append(a)
            keep['split'].append(split)
            keep['passages'].append(p)
            keep['to_get_rid_of'].append(rid_off)
print(len(keep['question_id']))

df = pd.DataFrame(keep)
#df.to_csv("/projects/iris/rferreir/datasets/MsMarco/data2.csv", index=False)

100%|██████████| 56721/56721 [00:00<00:00, 276616.49it/s]

30572


In [9]:
print(len(df[df['split'] == 'test']))

2907


In [10]:
from sklearn.model_selection import train_test_split

df_train = df[df["split"] == "train"]

# Split : 15% to dev 
train_part, dev_part = train_test_split(df_train, test_size=0.15, random_state=42)

df.loc[train_part.index, "split"] = "train"
df.loc[dev_part.index, "split"] = "dev"

In [11]:
df = df[df["to_get_rid_of"] == False]
df = df.drop(columns=["to_get_rid_of"])
print(df['split'].value_counts())

split
train    23513
dev       4149
test      2907
Name: count, dtype: int64


In [12]:
df.head()

,question_id,question,answer,split,passages
0,1089952,Brea is in what county ?,Brea is in Orange County.,test,"[{'is_selected': 0, 'passage_text': 'Brea Truc..."
1,1089525,Tollhouse ca is in what county ?,"Tollhouse is in Fresno County, California.",test,"[{'is_selected': 0, 'passage_text': 'tollhouse..."
2,1089298,Burnet texas is in what county ?,Burnet is a city in and the county seat of Bur...,test,"[{'is_selected': 0, 'passage_text': 'Burnet is..."
3,1089187,Union grove is in what county ?,Union Grove is in Iredell County and Racine Co...,test,"[{'is_selected': 0, 'passage_text': 'Village o..."
4,1089143,Upper darby is what county in pennsylvania ?,"Upper Darby is in Delaware County, Pennsylvani...",test,"[{'is_selected': 0, 'passage_text': 'Upper Dar..."


## To HF dataset

In [13]:
train_split = df[df['split']=='train'].drop(columns=['split'])
dev_split = df[df['split']=='dev'].drop(columns=['split'])
test_split = df[df['split']=='test'].drop(columns=['split'])

d_test = datasets.Dataset.from_pandas(test_split).remove_columns("__index_level_0__")
d_train = datasets.Dataset.from_pandas(train_split).remove_columns("__index_level_0__")
d_dev = datasets.Dataset.from_pandas(dev_split).remove_columns("__index_level_0__")

d1 = datasets.DatasetDict({
        "test": d_test,
        "train": d_train,
        "validation": d_dev
    })

print(d1)
print(d1['train'][0])
d1.save_to_disk(os.path.join(output_path, "MsMarco"))

DatasetDict({
    test: Dataset({
        features: ['question_id', 'question', 'answer', 'passages'],
        num_rows: 2907
    })
    train: Dataset({
        features: ['question_id', 'question', 'answer', 'passages'],
        num_rows: 23513
    })
    validation: Dataset({
        features: ['question_id', 'question', 'answer', 'passages'],
        num_rows: 4149
    })
})
{'question_id': 605123, 'question': 'What county is dewitt michigan in? ?', 'answer': 'DeWitt is located in Clinton County, Michigan, United States.', 'passages': [{'is_selected': 0, 'passage_text': 'DeWitt was named after DeWitt Clinton, Governor of New York during the 1820s. It was first settled by Captain David Scott, who moved there from Ann Arbor in 1833, and platted the land. The State Legislature formally created DeWitt Township on March 23, 1836.', 'url': 'https://en.wikipedia.org/wiki/DeWitt,_Michigan'}, {'is_selected': 0, 'passage_text': "Clinton County Michigan Sheriff's Office shared Clinton County 

Saving the dataset (1/1 shards): 100%|██████████| 4149/4149 [00:00<00:00, 55714.03 examples/s]


In [15]:
import hashlib
import json

d2 = datasets.load_dataset("rfr2003/GeoBenchLLM", "MsMarco")
# We check that the content of the dataset is the same
def content_hash(ds):
    h = hashlib.sha256()
    for ex in ds:
        h.update(json.dumps(ex, sort_keys=True).encode())
    return h.hexdigest()

for split in ['train', 'test', 'validation']:
    assert content_hash(d1[split]) == content_hash(d2[split])

Generating validation split: 100%|██████████| 4149/4149 [00:00<00:00, 31058.88 examples/s]
